# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2 — Refresh / Content Opportunity Scoring.**

The starter data (below) shows a large, capacity-constrained review queue: 43.8% of pages are declining with real demand behind them, spread across almost every client. That is far more candidates than any editor could review by hand, which is exactly the setup this lane is built for — a ranked queue with reason codes, not a report. The starter pipeline in this repo (`scripts/01`–`05`) is already built for this lane, so I have a working rule baseline and a trained model to compare against instead of starting from zero.

In [2]:
import os
import pandas as pd

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

n_clients = df["client_id"].nunique()
declining_with_demand = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
page_one_decay_risk = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
low_ctr_visible_page = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)

print(f"rows: {len(df):,} | clients: {n_clients}")
print(f"declining_with_demand : {declining_with_demand.sum():,} rows ({declining_with_demand.mean()*100:.1f}%)")
print(f"page_one_decay_risk   : {page_one_decay_risk.sum():,} rows ({page_one_decay_risk.mean()*100:.1f}%)")
print(f"low_ctr_visible_page  : {low_ctr_visible_page.sum():,} rows ({low_ctr_visible_page.mean()*100:.1f}%)")

rows: 30,000 | clients: 32
declining_with_demand : 13,152 rows (43.8%)
page_one_decay_risk   : 7,076 rows (23.6%)
low_ctr_visible_page  : 9,759 rows (32.5%)


## 2. The question: decision, action, cost of a wrong call

**Decision:** Given a client's existing content, which pages should the content/SEO team review first for refresh — and roughly why (reason code)?

**Who acts, and how:** A content editor or SEO lead with limited weekly review time. They take the ranked queue, open the top N pages, and choose to refresh, expand, protect, prune, or monitor each one based on its reason code.

**Cost of a wrong call:**
- *False positive* (flagged urgent, isn't really declining): wasted editor hours — expensive because candidates already outnumber review capacity (see the per-client count below).
- *False negative* (a real, high-demand decline never surfaces): a traffic-relevant decline goes unnoticed until it's worse. This is the costlier error, so the ranking should favor catching high-demand declines over being conservative.

**Why a plain rule isn't enough:** The starter baseline rules (`stale_visible_page`, `declining_with_demand`, etc.) already exist as simple if-statements, but each checks one signal at a time — they can't weigh "declining, but low position, but aging, but still gets clicks" as one combined risk. The repo's own baseline-vs-model comparison (below) shows that combined weighing already beats the flat rule on this data.

In [3]:
clients_touched = df.loc[declining_with_demand, "client_id"].nunique()
avg_per_client = declining_with_demand.sum() / n_clients
print(f"declining_with_demand touches {clients_touched} of {n_clients} clients")
print(f"average candidates per client: {avg_per_client:.0f}")

import json

with open("outputs/model_results.json") as f:
    results = json.load(f)

baseline_p50 = results["baseline"]["baseline_precision_at_50"]
rf_p50 = results["models"]["random_forest"]["precision_at_50"]
print(f"baseline rule precision@50: {baseline_p50:.3f}")
print(f"random forest precision@50: {rf_p50:.3f}  ({rf_p50 / baseline_p50:.1f}x the baseline)")

declining_with_demand touches 29 of 32 clients
average candidates per client: 411
baseline rule precision@50: 0.240
random forest precision@50: 0.680  (2.8x the baseline)


## 3. Quick look at the data (2-3 real numbers)

Loaded `data/raw/content_refresh_anonymized.csv` above (30,000 rows, 32 clients). The numbers that justify this lane:

1. **43.8%** of pages (13,152 / 30,000) are declining with real demand behind them (`trend_direction == "down"` and `impressions_90d >= 100`).
2. **23.6%** of pages (7,076) already rank top-10 but are aging (`content_age_days >= 180`) — decay risk on pages that currently work.
3. Candidates land on **29 of 32 clients**, averaging **~411 declining-with-demand pages per client** — too many to review by hand, which is the case for a ranked queue over a flat list.
4. On this same data, a learned model already beats the flat rule at ranking: precision@50 goes from 0.240 (rule) to 0.680 (random forest) — see the exact numbers above, pulled from this repo's own `outputs/model_results.json`.

In [4]:
summary = pd.Series({
    "declining_with_demand": declining_with_demand.sum(),
    "page_one_decay_risk": page_one_decay_risk.sum(),
    "low_ctr_visible_page": low_ctr_visible_page.sum(),
    "overlap (declining & page-1 decay)": (declining_with_demand & page_one_decay_risk).sum(),
})
summary

declining_with_demand                 13152
page_one_decay_risk                    7076
low_ctr_visible_page                   9759
overlap (declining & page-1 decay)     2827
dtype: int64

## 4. Careful words: what I can and can't claim

**What this work can say:**
- *Observed*: which pages show declining traffic, thinning engagement, or position decay in the trailing 90 days.
- *Directional / decision-support*: a ranked queue that puts the highest-risk, highest-demand pages first, with reason codes a reviewer can check by hand.
- On this starter slice, under client-holdout validation, a learned ranking beats the fixed rule: precision@50 = 0.240 (rule) vs 0.680 (random forest) — roughly 2.8x more true positives in the top 50 picks. That's evidence for using ML here, not just an assumption.
- The model's top signals (`days_with_impressions`, `log_impressions_90d`, `avg_position`, `content_age_days`) are all plain observed metrics — no hidden product score is doing the work.

**What it can never say:**
- Not causal. A page flagged for refresh is not guaranteed to recover if edited — proving that needs an experiment (e.g. a controlled before/after refresh test), not this dataset.
- Not a Google-ranking-factor discovery. `avg_position` and friends are observed outcomes, not evidence of *why* Google ranks a page where it does.
- The starter target (`is_declining_label`) is a **proxy label** — computed from the current window's `trend_direction`, not a true future outcome. A stronger capstone version predicts decline/recovery over a *future* 30-day window using only *prior* 90-day features. I'll aim to move to that future-window label as the project matures (by end of Week 4 at the latest).

In [5]:
for feat in results["best_model"]["feature_importance_top"][:4]:
    print(f"{feat['feature']:<22} {feat['importance']:.3f}")

days_with_impressions  0.161
log_impressions_90d    0.128
avg_position           0.108
content_age_days       0.095


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.